# Entrainement LoCon SDXL — personnage "Solle" (Illustrious XL 0.1)

Notebook **Kaggle** (GPU T4 16 Go gratuit) adapte a ton setup : base **Illustrious XL 0.1**, type **LoCon** (dim 16), dataset de **79 images** (poses/angles/expressions variees), trigger **sollechar**.

## Avant de lancer
1. **Cree un Dataset Kaggle prive** contenant le contenu de ton dossier local `training/dataset/`
   (les 79 `.png` + 79 `.txt`). Nomme-le par ex. `solle-dataset-v2`.
2. Menu de droite **Add Input** -> ajoute `solle-dataset-v2`. Il sera monte sous `/kaggle/input/solle-dataset-v2/`.
3. Menu de droite : **Accelerator = GPU T4 x2** (ou P100), **Internet = ON**.
4. Si ton dataset a un autre nom, change la variable `SRC` dans la cellule 4.
5. Execute les cellules de haut en bas.

A la fin, telecharge `solle_locon.safetensors` (onglet Output) et remplace `solle_img-10.safetensors` dans `generate_lora.py`.

## 1. Verifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer sd-scripts (moteur kohya)

In [ ]:
%cd /kaggle/working
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
# Version figee connue pour fonctionner avec SDXL
!git checkout v0.8.7
!pip install -q -r requirements.txt
!pip install -q xformers bitsandbytes
# accelerate : 1 GPU, fp16 (T4 = pas de bf16)
import os, textwrap
os.makedirs(os.path.expanduser('~/.cache/huggingface/accelerate'), exist_ok=True)
cfg = textwrap.dedent('''\
    compute_environment: LOCAL_MACHINE
    distributed_type: 'NO'
    mixed_precision: fp16
    num_processes: 1
    use_cpu: false
''')
open(os.path.expanduser('~/.cache/huggingface/accelerate/default_config.yaml'),'w').write(cfg)
print('OK')

## 3. Telecharger le modele de base Illustrious XL 0.1

In [ ]:
%cd /kaggle/working
!mkdir -p models
# Illustrious XL 0.1 (~6.9 Go) — MEME base que ton LoRA actuel
!wget -q -O models/illustrious_xl_v0.1.safetensors \
  https://huggingface.co/OnomaAIResearch/Illustrious-xl-early-release-v0/resolve/main/Illustrious-XL-v0.1.safetensors
!ls -lh models/

## 4. Preparer le dataset

kohya attend une structure `<dossier>/<repeats>_<nom>/` avec images + .txt.
Prefixe `5_` = 5 repetitions/image. 79 images x 5 x 10 epochs / batch 2 = **~1975 steps**.

In [ ]:
import os, glob, shutil

SRC = '/kaggle/input/solle-dataset-v2'        # <-- adapte si ton dataset a un autre nom
DST = '/kaggle/working/train_data/5_sollechar'
os.makedirs(DST, exist_ok=True)

n = 0
for f in glob.glob(os.path.join(SRC, '*')):
    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.txt')):
        shutil.copy(f, DST)
        n += 1
imgs = len(glob.glob(DST + '/*.png')) + len(glob.glob(DST + '/*.jpg'))
txts = len(glob.glob(DST + '/*.txt'))
print(f'{n} fichiers copies | images={imgs} captions={txts}')
assert imgs == txts, 'Mismatch images/captions !'
print('Exemple caption :', open(sorted(glob.glob(DST + '/*.txt'))[0]).read())

## 5. Lancer l'entrainement (LoCon, dim 16)

~2000 steps, rank 16 + couches conv (LoCon). Sur T4 compte ~1h30-2h30.
Le LoCon est sauve toutes les 2 epochs (garde l'avant-derniere si la derniere sur-apprend).

In [ ]:
%cd /kaggle/working/sd-scripts
!accelerate launch --num_cpu_threads_per_process 2 sdxl_train_network.py \
  --pretrained_model_name_or_path=/kaggle/working/models/illustrious_xl_v0.1.safetensors \
  --train_data_dir=/kaggle/working/train_data \
  --output_dir=/kaggle/working/output \
  --output_name=solle_locon \
  --resolution=1024,1024 \
  --enable_bucket --min_bucket_reso=640 --max_bucket_reso=1536 \
  --network_module=networks.lora \
  --network_dim=16 \
  --network_alpha=8 \
  --network_args "conv_dim=16" "conv_alpha=8" \
  --train_batch_size=2 \
  --gradient_checkpointing \
  --max_train_epochs=10 \
  --learning_rate=3e-4 \
  --unet_lr=3e-4 \
  --text_encoder_lr=6e-5 \
  --lr_scheduler=cosine_with_restarts \
  --lr_scheduler_num_cycles=3 \
  --lr_warmup_steps=100 \
  --optimizer_type=AdamW8bit \
  --optimizer_args "weight_decay=0.1" "betas=0.9,0.99" \
  --mixed_precision=fp16 \
  --save_precision=fp16 \
  --min_snr_gamma=5 \
  --cache_latents --cache_latents_to_disk \
  --save_model_as=safetensors \
  --save_every_n_epochs=2 \
  --caption_extension=.txt \
  --shuffle_caption --keep_tokens=2 \
  --max_data_loader_n_workers=2 \
  --seed=42 \
  --xformers \
  --no_half_vae

## 6. Recuperer le LoCon

Fichiers dans `/kaggle/working/output/`. Ouvre l'onglet **Output** (panneau de droite) pour telecharger.
Garde aussi une version intermediaire (`solle_locon-000008.safetensors` etc.) au cas ou la derniere epoch sur-apprend.

In [ ]:
!ls -lh /kaggle/working/output/